In [1]:
import joblib
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler


# Temizlenmiş Veri Setini Yükleme
bundle = joblib.load("../backend/models/cancer/cancer_data_bundle.pkl")

X_train = bundle["X_train"]
X_test = bundle["X_test"]
y_train = bundle["y_train"]
y_test = bundle["y_test"]

print("Veriler başarıyla yüklendi!")
print(f"X_train boyutu: {X_train.shape}, X_test boyutu: {X_test.shape}")

Veriler başarıyla yüklendi!
X_train boyutu: (455, 5), X_test boyutu: (114, 5)


In [2]:
svm_model = SVC(
    kernel="rbf",
    gamma="scale",
    C=1,
    degree=4,
    random_state=42,
)

In [3]:
# ÖLÇEKLENDİRME
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [4]:
# K-Fold Cross Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(svm_model, X_train, y_train, cv=cv, scoring="accuracy")

print("K-Fold (5 Katlı) Doğrulama Skorları:", cv_scores)
print("Ortalama CV Doğruluğu: {:.4f} (+/- {:.4f})".format(cv_scores.mean(), cv_scores.std() * 2))

K-Fold (5 Katlı) Doğrulama Skorları: [0.94505495 0.96703297 0.89010989 0.9010989  0.92307692]
Ortalama CV Doğruluğu: 0.9253 (+/- 0.0563)


In [5]:
# Tüm Eğitim Verisiyle Modeli Son Kez Eğitme
svm_model.fit(X_train, y_train)
print("\nModel tüm eğitim setiyle son kez eğitildi!")


Model tüm eğitim setiyle son kez eğitildi!


In [6]:
# Train Seti Performansı (Overfitting kontrolü için)
y_train_pred = svm_model.predict(X_train)
train_accuracy = accuracy_score(y_train, y_train_pred)

print("=" * 40)
print("          TRAIN SETİ PERFORMANSI          ")
print("=" * 40)
print(f"Train Doğruluk Oranı (Accuracy): {train_accuracy:.4f}\n")
print("Train Sınıflandırma Raporu (Precision, Recall, F1-Score):")
print(classification_report(y_train, y_train_pred))

# Test Seti Performansı (Gerçek başarı)
y_test_pred = svm_model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_test_pred)

print("=" * 40)
print("          TEST SETİ PERFORMANSI           ")
print("=" * 40)
print(f"Test Doğruluk Oranı (Accuracy): {test_accuracy:.4f}\n")
print("Test Sınıflandırma Raporu (Precision, Recall, F1-Score):")
print(classification_report(y_test, y_test_pred))

          TRAIN SETİ PERFORMANSI          
Train Doğruluk Oranı (Accuracy): 0.9429

Train Sınıflandırma Raporu (Precision, Recall, F1-Score):
              precision    recall  f1-score   support

           0       0.94      0.97      0.96       285
           1       0.94      0.90      0.92       170

    accuracy                           0.94       455
   macro avg       0.94      0.93      0.94       455
weighted avg       0.94      0.94      0.94       455

          TEST SETİ PERFORMANSI           
Test Doğruluk Oranı (Accuracy): 0.9298

Test Sınıflandırma Raporu (Precision, Recall, F1-Score):
              precision    recall  f1-score   support

           0       0.93      0.96      0.95        72
           1       0.93      0.88      0.90        42

    accuracy                           0.93       114
   macro avg       0.93      0.92      0.92       114
weighted avg       0.93      0.93      0.93       114



In [7]:
# Nihai Modeli Backend Altına Kaydetme
joblib.dump(svm_model, "../backend/models/cancer/svm_model.pkl")
print("Eğitilmiş model başarıyla kaydedildi!")

Eğitilmiş model başarıyla kaydedildi!


In [ ]:
# 1. Orijinal ham CSV dosyasını doğrudan oku
df = pd.read_csv("../datasets/cancer_data.csv")
# 2. X ve y olarak ayır
X = df.drop(
    ["id","diagnosis","Unnamed: 32"], axis=1
)  # Kendi hedef sütun adını yazmalısın
y = df["diagnosis"]

# 3. Train-Test olarak böl (Analiz notebook'undaki test_size ve random_state AYNİ olmalı!)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Ham veri yüklendi ve bölündü!")

# 4. Modeli ham veriyle eğit
svm_raw_model = SVC(
    kernel="rbf",
    gamma="scale",
    C=1,
    degree=4,
    random_state=42,
)
svm_raw_model.fit(X_train_raw, y_train)

# 5. Ham Model İçin Train Seti Performansı
y_train_pred_raw = svm_raw_model.predict(X_train_raw)
train_accuracy_raw = accuracy_score(y_train, y_train_pred_raw)

print("=" * 40)
print("      HAM VERİ - TRAIN SETİ PERFORMANSI      ")
print("=" * 40)
print(f"Train Doğruluk Oranı (Accuracy): {train_accuracy_raw:.4f}\n")
print("Train Sınıflandırma Raporu (Precision, Recall, F1-Score):")
print(classification_report(y_train, y_train_pred_raw))

# 6. Ham Model İçin Test Seti Performansı (Gerçek başarı)
y_test_pred_raw = svm_raw_model.predict(X_test_raw)
test_accuracy_raw = accuracy_score(y_test, y_test_pred_raw)

print("=" * 40)
print("      HAM VERİ - TEST SETİ PERFORMANSI       ")
print("=" * 40)
print(f"Test Doğruluk Oranı (Accuracy): {test_accuracy_raw:.4f}\n")
print("Test Sınıflandırma Raporu (Precision, Recall, F1-Score):")
print(classification_report(y_test, y_test_pred_raw))